In [32]:
import math

# Verify Theorem 4.4


Here is a simple introduction to your implementation of basic arithmetic operations in a quadratic algebra. In this code, an element $a + b\sqrt{D}$ over a base ring $R$ is represented by the tuple `(a, b)`.

### 1. `mul` (Multiplication)
This function computes the product of two elements. By expanding the product mathematically:
$$(a_1 + b_1\sqrt{D})(a_2 + b_2\sqrt{D}) = (a_1a_2 + b_1b_2D) + (a_1b_2 + b_1a_2)\sqrt{D}$$
The code simply groups and returns the resulting real and imaginary components.

### 2. `square` (Squaring)
This function computes the square of an element. It serves as an optimized version of multiplication where both operands are identical:
$$(a + b\sqrt{D})^2 = (a^2 + b^2D) + 2ab\sqrt{D}$$

### 3. `Reduce` (Division)
This function computes the division of two elements $\frac{a + b\sqrt{D}}{c + d\sqrt{D}}$ by rationalizing the denominator. It multiplies both the numerator and the denominator by the conjugate $(c - d\sqrt{D})$:
$$\frac{(a + b\sqrt{D})(c - d\sqrt{D})}{(c + d\sqrt{D})(c - d\sqrt{D})} = \frac{(ac - bdD) + (bc - ad)\sqrt{D}}{c^2 - d^2D}$$
In the code, `temp` calculates the inverse of the norm $(c^2 - d^2D)^{-1}$ over the base ring. The real part $(ac - bdD)$ and imaginary part $(bc - ad)$ are then multiplied by this inverse.

### 4. `Exp` (Exponentiation)
This function computes the power $(a + b\sqrt{D})^{\text{exp}}$ using the highly efficient **Right-to-Left Square-and-Multiply algorithm**. 
* It converts the exponent into a binary string.
* It iterates through the binary string from the Least Significant Bit (LSB) to the Most Significant Bit (MSB).
* If the current bit is $1$, it multiplies the running base (`powerA, powerB`) into the result (`resultA, resultB`).
* It squares the base in every iteration to prepare the correct weight for the next bit, ensuring the operation is completed in logarithmic time rather than linear time.

In [1]:
def Reduce(a,b,c,d, D, R):
    temp = R(c^2-d^2*D)^(-1)
    outputA = R(a*c-b*d*D)*temp
    outputB = R(b*c-a*d)*temp
    return outputA, outputB

def square(a,b, D, R):
    return R(a^2+b^2*D), R(2*a*b)

def mul(a1,b1, a2, b2, D, R):
    return R(a1*a2+b1*b2*D), R(a1*b2+b1*a2)

def Exp(a,b,D,R,exp):
    temp = "{0:b}".format(exp)
    resultA, resultB = R(1), R(0)
    powerA, powerB = a, b
    for i in range(len(temp)-1, -1, -1):
        bit = int(temp[i])
        if bit == 1:
            resultA, resultB = mul(resultA, resultB, powerA, powerB, D, R)
        powerA, powerB = square(powerA, powerB, D, R)
    return resultA, resultB

## Lemma 4.3

The implementation is designed to directly evaluate the following theoretical formula:

Let $n = \prod_{i=1}^s p_i^{r_i}$ be a positive odd integer. If $D$ is chosen such that $\left(\frac{-D}{n}\right)= -1$, then one has:

$$|{\rm Vset}(D,n)|=\prod_{i=1}^s p_i^{\lfloor r_i/2 \rfloor}\big( \gcd(e,d_i)-1\big)+\prod_{i=1}^sp_i^{\lfloor r_i/2 \rfloor} \gcd(e,d_i)$$

where $p_i-\left(\frac{D}{p_i}\right)=2^{k_i}d_i$ with $d_i$ odd, and $e = \big(n +\big(\frac{-1}{n}\big) \big)/2.$

In [33]:
def GetTSLCount(n):
    R = Integers(n)
    
    # Phase 1: Parameter Setup
    # Deterministically find the first D satisfying the strict Jacobi constraint (-D/n) = -1.
    # This globally ensures a fixed-length evaluation path and intrinsically filters perfect squares.
    D = 1
    while kronecker(-D, n) != -1:
        D += 1
    
    # The fixed-length exponent e = (n + (-1/n)) / 2.
    # This completely bypasses the vulnerable variable-length 2^s * d decomposition.
    e = (n + kronecker(-1, n)) // 2
    inv2 = R(2)^(-1)
    inv4 = R(4)^(-1)
    
    liar_count = 0    # Corresponds to the number of false witnesses: |TSL(D, n)|
    sample_space = 0  # Corresponds to the valid cryptographic sampling space: |Z(D, n)|
    
    # Phase 2: Exhaustive Search over the Parameter Space
    for P in range(n):
        # Enforce the fundamental Lucas parameter relation: P^2 - 4Q == D (mod n)
        Q = R(P^2 - D) * inv4
        
        # Domain Constraint: Ensure Q is coprime to n to form a valid Z(D, n) space.
        if gcd(int(Q), n) == 1:
            sample_space += 1
            
            # Construct the core element: tau = (P + sqrt(D)) / (P - sqrt(D))
            # Here, tau represents alpha * beta^(-1) in the quadratic extension ring O_D / n O_D.
            # We represent the fraction (a + b*sqrt(D)) / (c + d*sqrt(D)) as follows:
            # Numerator: P + 1*sqrt(D)   => a = P, b = 1
            # Denominator: P - 1*sqrt(D) => c = P, d = -1
            a = R(P) 
            b = 1
            c = R(P) 
            d = R(-1) 
            
            # Reduce() handles the modular division to compute tau = u + v*sqrt(D)
            u, v = Reduce(a, b, c, d, D, R)
            
            # Evaluate the fixed-length exponentiation: tau^e = X + Y*sqrt(D) (mod n O_D)
            X, Y = Exp(u, v, D, R, e)
            
            # Phase 3: The Accelerated Trace Verification (Algorithm CV)
            # The vTraceLucas test verifies if Trace(tau^e) == 2 or -2 (mod n).
            # Since tau^e = X + Y*sqrt(D), its Trace is simply (X + Y*sqrt(D)) + (X - Y*sqrt(D)) = 2X.
            # Therefore, Trace(tau^e) == +/- 2 is algebraically equivalent to checking X == +/- 1.
            # This elegant property allows us to entirely ignore the imaginary part (Y).
            if X == R(1) or X == R(-1):
                liar_count += 1
                
    return liar_count, sample_space

The following implementation calculates the exact cardinality of the set ${\rm Vset}(D,n)$ and the total sample space $\mathcal{Z}(D,n)$ by directly applying their theoretical definitions and mathematical formulas.

In [34]:
def CalculateTheoreticalTSL(n):
    # 1. Find D such that (-D/n) = -1
    D = 1
    while kronecker(-D, n) != -1:
        D += 1
        
    e = (n + kronecker(-1, n)) // 2
    factors = factor(n)
    
    Z_count = 1
    
    # According to your formula, V(D,n) is the sum of two independent products
    # prod_plus_1  corresponds to \prod p^{\lfloor r/2 \rfloor} \gcd(e, d_i)
    # prod_minus_1 corresponds to \prod p^{\lfloor r/2 \rfloor} (\gcd(e, d_i) - 1)
    prod_plus_1 = 1   
    prod_minus_1 = 1  
    
    for p_val, r_val in factors:
        p = int(p_val)
        r = int(r_val)
        
        # --- Calculate the product term for the sample space |Z(D,n)| ---
        leg = kronecker(D, p)
        # Formula: p^{r-1} * (p - (D/p) - 1)
        Z_count *= (p**(r - 1)) * (p - leg - 1)
        
        # --- Calculate the product term for |V(D,n)| ---
        # Extract the odd part of p - (D/p) as d_i
        temp = p - leg
        while temp % 2 == 0:
            temp //= 2
        d_i = temp
        
        gcd_val = gcd(e, d_i)
        lift_factor = p**(r // 2)
        
        # Multiply into the two independent products respectively
        prod_plus_1 *= lift_factor * gcd_val
        prod_minus_1 *= lift_factor * (gcd_val - 1)
        
    V_count = prod_plus_1 + prod_minus_1
    
    return V_count, Z_count

## Verify: Theorem and Definition

This code snippet concretely verifies the accuracy of the theoretical formula by comparing its output against the empirical results obtained directly from the definition.It iterates through composite numbers up to a specified bound (skipping perfect squares and primes) and computes both the theoretical counts (CalculateTheoreticalTSL) and the empirical counts (GetTSLCount). By asserting that the two values perfectly match for every valid $n$, it proves that the theoretical formula correctly and reliably calculates the exact cardinality of the sets.

In [31]:
Bound = 100  

for i in range(3, Bound, 2):
    # The condition (-D/n) = -1 is impossible if n is a perfect square.
    root = math.isqrt(i)
    if root * root == i:
        continue

    # (Soundness Error)
    if is_prime(i):
        continue

    # Calculate theoretical and empirical values
    a = CalculateTheoreticalTSL(i)
    b = GetTSLCount(i)
    
    # Verify if they match perfectly
    if a != b:
        print(f"❌ Mismatch at n={i}! Theory={a}, Empirical={b}")
    else:
        # Print the composite n, and its (number of Liars, total sample space)
        print(f"✅ Match at n={i}! (V, Z) = {a}")

✅ Match at n=15! (V, Z) = (1, 3)
✅ Match at n=21! (V, Z) = (1, 15)
✅ Match at n=27! (V, Z) = (3, 9)
✅ Match at n=33! (V, Z) = (1, 27)
✅ Match at n=35! (V, Z) = (1, 15)
✅ Match at n=39! (V, Z) = (1, 11)
✅ Match at n=45! (V, Z) = (3, 45)
✅ Match at n=51! (V, Z) = (1, 15)
✅ Match at n=55! (V, Z) = (1, 27)
✅ Match at n=57! (V, Z) = (1, 51)
✅ Match at n=63! (V, Z) = (3, 15)
✅ Match at n=65! (V, Z) = (13, 55)
✅ Match at n=69! (V, Z) = (1, 63)
✅ Match at n=75! (V, Z) = (5, 15)
✅ Match at n=77! (V, Z) = (13, 55)
✅ Match at n=85! (V, Z) = (1, 75)
✅ Match at n=87! (V, Z) = (1, 27)
✅ Match at n=91! (V, Z) = (13, 55)
✅ Match at n=93! (V, Z) = (1, 87)
✅ Match at n=95! (V, Z) = (1, 51)
✅ Match at n=99! (V, Z) = (3, 27)


# F. EMPIRICAL DISTRIBUTION OF PARAMETER D SEARCH DEPTH

This implementation executes the statistical simulation described in your text, designed to empirically test how quickly and reliably we can find a valid parameter $D$. 

Here is a breakdown of how the code achieves this:

### 1. Fast Prime Generation (`generate_fast_crypto_prime`)
To run a massive simulation (like testing 990,000 candidates) in a reasonable amount of time, this helper function simulates the high-speed prime generation typically found in low-level languages like C++ or Golang. It repeatedly generates random odd integers of the target bit length and filters them using the highly optimized BPSW primality test (via `is_pseudoprime()`) to yield probable primes.

### 2. The Core Benchmark (`findMinimalNonResidueFast`)
This function orchestrates the actual simulation and tracks the performance metrics:

* **Sequential Search:** For every generated prime candidate $n$, it loops through a pre-calculated list of small odd primes $D \in \{3, 5, 7, \dots\}$.
* **Condition Checking:** It evaluates the Jacobi symbol condition $\left(\frac{-D}{n}\right) = -1$ using the `kronecker(-p, n)` function.
* **Tracking Depth:** Once a valid $D$ is found, it records exactly how many iterations it took (the "search depth") and immediately breaks the loop to move on to the next candidate $n$. It also continuously updates the absolute maximum search depth and the largest prime $D$ ever needed across the entire simulation.

### 3. Statistical Reporting
After processing the specified number of prime candidates, the script aggregates the data and prints a comprehensive report. It calculates the total execution time and outputs a precise distribution showing how frequently each search depth occurred (both in raw counts and percentages). This final output provides the concrete empirical evidence needed to prove the efficiency and completeness of the fixed-iteration search strategy.

In [21]:
def generate_fast_crypto_prime(bitLength):
    """Ultra-fast generation of a probable prime (simulating C++/Golang behavior)"""
    lower_bound = 2**(bitLength - 1)
    upper_bound = 2**bitLength - 1
    
    while True:
        # Generate a random integer and force it to be odd
        n = ZZ.random_element(lower_bound, upper_bound)
        if n % 2 == 0:
            n += 1
            
        # is_pseudoprime() calls PARI's BPSW test under the hood, which is extremely fast!
        if n.is_pseudoprime():
            return n

def findMinimalNonResidueFast(bitLength, countofPrime):
    primes_L = [nth_prime(i) for i in range(2, 27)]
    
    max_search_depth = 0
    max_D_used = 0
    depth_distribution = {i: 0 for i in range(1, 27)}
    
    print(f"🚀 Starting ultra-fast benchmark: testing {countofPrime} {bitLength}-bit primes...")
    
    import time
    start_time = time.time()
    
    for i in range(countofPrime):
        if (i + 1) % 1000 == 0:
            print(f"...Completed {i + 1} tests...")
            
        # Use our custom ultra-fast generator
        n = generate_fast_crypto_prime(bitLength)
        
        found = False
        iterations = 0
        
        for p in primes_L:
            iterations += 1
            # Check Jacobi symbol (-D/n) == -1
            if kronecker(-p, n) == -1:
                found = True
                depth_distribution[iterations] += 1
                if iterations > max_search_depth:
                    max_search_depth = iterations
                    max_D_used = p
                break
                
    elapsed = time.time() - start_time
    
    print("\n=== 📊 Parameter D Search Depth Statistical Report ===")
    print(f"Total time elapsed: {elapsed:.2f} seconds")
    print(f"Total tests: {countofPrime}")
    print(f"Maximum search depth (Max Iterations): {max_search_depth}")
    print(f"Maximum prime D used: {max_D_used}")
    print("\n[Search Depth Distribution]")
    for depth, count in depth_distribution.items():
        if count > 0:
            # Add float() to force cast Sage's rational numbers to ensure no formatting errors
            probability = float(count / countofPrime) * 100
            # Add int() to ensure the value of D is a standard Python integer
            D_val = int(primes_L[depth-1])
            print(f"Requires {depth:2d} iterations (D={D_val:3d}): {count:6d} times ({probability:5.2f}%)")

In [22]:
findMinimalNonResidueFast(1024,10)

🚀 Starting ultra-fast benchmark: testing 10 1024-bit primes...

=== 📊 Parameter D Search Depth Statistical Report ===
Total time elapsed: 0.99 seconds
Total tests: 10
Maximum search depth (Max Iterations): 6
Maximum prime D used: 17

[Search Depth Distribution]
Requires  1 iterations (D=  3):      6 times (60.00%)
Requires  3 iterations (D=  7):      2 times (20.00%)
Requires  4 iterations (D= 11):      1 times (10.00%)
Requires  6 iterations (D= 17):      1 times (10.00%)


# G. Optimal Parameter Selection via Integer Programming

This Python code is a direct implementation of the two Bivariate Integer Linear Programming (ILP) solver algorithms presented in the paper. 

Specifically:

* **`solve_hvl_params`** implements **Algorithm 10 (ILP Solver for HVLTest Parameters)**. It calculates the optimal number of computationally intensive iterations ($x$) and trace-based iterations ($y$) to minimize the total arithmetic cost ($c_x \cdot x + c_y \cdot y$) for the HVL test, ensuring the target security level $\kappa$ is strictly met.
* **`solve_hvss_params`** implements **Algorithm 11 (ILP Solver for HVSSTest Parameters)**. It performs the same optimization but applies the specific entropy bounds and scaling factors required for the HVSS test.

Both functions execute an efficient $\mathcal{O}(\kappa)$ linear scan over the possible values of $x$. For each step, they calculate the minimum required $y$ to satisfy the security threshold against both square-free and non-square-free composites. The search terminates early as soon as $y$ drops to 0, outputting the optimal iteration pair `(opt_x, opt_y)`.

In [7]:
def solve_hvl_params(kappa: int, p_min: float, c_x: float, c_y: float) -> tuple:
    """
    ILP Solver for HVL Test Parameters
    """
    # Calculate H_u and H_v
    H_u = -math.log2(0.25 + 0.91 / (p_min - 2))
    H_v = -math.log2(0.5 + 1.2 / (p_min - 2))
    
    min_cost = float('inf')
    opt_x, opt_y = 0, 0
    
    # Precompute log2(p_min)
    log2_pmin = math.log2(p_min)
    
    # Loop from 0 to kappa (inclusive)
    for x in range(int(kappa) + 1):
        y_sf = max(0, math.ceil(kappa / H_u) - x)
        y_nsf = max(0, math.ceil((kappa - x * log2_pmin) / H_v))
        y = max(y_sf, y_nsf)
        
        cost = c_x * x + c_y * y
        
        if cost < min_cost:
            min_cost = cost
            opt_x, opt_y = x, y
            
        if y == 0:
            break
            
    return opt_x, opt_y

def solve_hvss_params(kappa: int, p_min: float, c_x: float, c_y: float) -> tuple:
    """
    ILP Solver for HVSS Test Parameters
    """
    # Calculate H_u and H_v
    H_u = -math.log2(0.25 + 0.91 / (p_min - 2))
    H_v = -math.log2(0.5 + 1.2 / (p_min - 2))
    
    min_cost = float('inf')
    opt_x, opt_y = 0, 0
    
    # Precompute log2(p_min) - 1 
    log2_pmin_minus_1 = math.log2(p_min) - 1
    
    # Loop from 0 to kappa (inclusive)
    for x in range(int(kappa) + 1):
        y_sf = max(0, math.ceil((kappa - x) / H_u))
        y_nsf = max(0, math.ceil((kappa - x * log2_pmin_minus_1) / H_v))
        y = max(y_sf, y_nsf)
        
        cost = c_x * x + c_y * y
        
        if cost < min_cost:
            min_cost = cost
            opt_x, opt_y = x, y
            
        if y == 0:
            break
            
    return opt_x, opt_y

In [ ]:
solve_hvl_params(128, 2647, 8.0, 2.0), solve_hvl_params(128, 101, 8.0, 2.0), solve_hvl_params(128, 503, 8.0, 2.0), solve_hvl_params(128, 1009, 8.0, 2.0)

((6, 60), (11, 57), (8, 57), (7, 59))

In [19]:
solve_hvss_params(128, 2647, 2.0, 2.0), solve_hvss_params(128, 101, 2.0, 2.0), solve_hvss_params(128, 503, 2.0, 2.0), solve_hvss_params(128, 1009, 2.0, 2.0)

((7, 61), (13, 60), (9, 60), (8, 61))

In [ ]:
solve_hvl_params(192, 2647, 8.0, 2.0), solve_hvss_params(192, 2647, 2.0, 2.0)

((9, 90), (10, 92))